# 第08章：Triton 注意力 GEMM — Decode 与 Prefill

## 本章内容

深入 vLLM 的 **Triton 注意力 kernel** 中的矩阵乘法：

1. 注意力中的 GEMM 与投影层 GEMM 的本质区别
2. Prefill: FlashAttention 风格的 tiled Q*K + attn*V
3. Decode: Flash-Decoding 的两阶段拆分
4. GQA Grouped Kernel: 将向量运算升级为矩阵运算
5. MLA 的 BLOCK_DPE: rope/nope 分离的双 tl.dot

## 前置知识
- Notebook 06-07 的矩阵乘法全景和 MoE GEMM
- FlashAttention 的基本原理（在线 softmax）
- KV Cache 和 Paged Attention 的概念

## 8.1 注意力 GEMM 的特殊性

注意力中的矩阵乘法与投影层的 GEMM 有根本区别：

```
投影 GEMM (如 Q_proj):
  C = A × B
  A: [num_tokens, 4096]     ← tokens 多, K 大
  B: [4096, 4096]           ← 固定权重
  → 标准大 GEMM, compute-bound

注意力 GEMM (Q*K^T):
  S = Q × K^T
  Q: [seq_len, 128]         ← head_dim 小 (64-192)
  K: [seq_len, 128]^T       ← K 来自 KV cache
  → head_dim 做 "K 维度", 非常小!
  → 不能独立做 GEMM — 必须和 softmax, attn*V 融合

为什么必须融合？
  S = Q × K^T  →  shape [seq_len, seq_len]
  对于 seq_len=128K, S 需要 128K × 128K × 2B = 32 GB 内存!
  → FlashAttention: 分块计算, 不存 S
```

In [ ]:
import torch
import triton
import triton.language as tl
import math

torch.manual_seed(42)
device = 'cuda'

# 注意力参数
batch_size = 4
num_heads = 32
head_dim = 128
seq_len = 1024

print("注意力 GEMM 的矩阵形状:")
print(f"  Q*K^T: [{seq_len}, {head_dim}] × [{head_dim}, {seq_len}] → [{seq_len}, {seq_len}]")
print(f"  attn*V: [{seq_len}, {seq_len}] × [{seq_len}, {head_dim}] → [{seq_len}, {head_dim}]")
print()
print(f"  head_dim = {head_dim} (远小于投影层的 K=4096)")
print(f"  seq_len = {seq_len} (决定中间张量大小)")
print(f"  中间张量 S: {seq_len}×{seq_len}×2 / 1024**2:.1f} MB / head")
print(f"  如果 seq_len=128K: {128000**2 * 2 / 1024**3:.1f} GB / head → 必须用 FlashAttention!")

## 8.2 Prefill: FlashAttention 风格的 Triton 实现

Prefill 阶段处理整个 prompt，Q 和 K 都有多个 token。
vLLM 的 `triton_prefill_attention.py` 实现了 FlashAttention 风格的分块计算。

### 核心算法: Online Softmax + Tiled GEMM

$$o_i = \frac{\sum_j e^{s_{ij} - m_i} v_j}{\sum_j e^{s_{ij} - m_i}}$$

其中 $s_{ij} = q_i \cdot k_j / \sqrt{d_k}$，$m_i = \max_j s_{ij}$

In [ ]:
@triton.jit
def flash_attention_prefill_kernel(
    Q, K, V, Out,
    sm_scale,
    stride_qb, stride_qh, stride_qm, stride_qd,
    stride_kb, stride_kh, stride_kn, stride_kd,
    stride_vb, stride_vh, stride_vn, stride_vd,
    stride_ob, stride_oh, stride_om, stride_od,
    seq_len,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_D: tl.constexpr,
    IS_CAUSAL: tl.constexpr,
):
    """
    FlashAttention-style Triton kernel (简化版).
    对应 vLLM: vllm/v1/attention/ops/triton_prefill_attention.py

    Grid: (batch, num_heads, cdiv(seq_len, BLOCK_M))
    """
    batch_id = tl.program_id(0)
    head_id = tl.program_id(1)
    block_m_id = tl.program_id(2)

    # Q block 的 token 偏移
    offs_m = block_m_id * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_D)

    # 加载 Q block: [BLOCK_M, BLOCK_D]
    q_ptrs = (Q + batch_id * stride_qb + head_id * stride_qh +
              offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd)
    q = tl.load(q_ptrs, mask=(offs_m[:, None] < seq_len) & (offs_d[None, :] < BLOCK_D),
                other=0.0)

    # Online softmax 状态
    m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")  # max
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)                  # sum(exp)
    acc = tl.zeros([BLOCK_M, BLOCK_D], dtype=tl.float32)         # output

    # 确定 K/V 的遍历范围
    end_n = seq_len
    if IS_CAUSAL:
        end_n = min(end_n, (block_m_id + 1) * BLOCK_M)

    # ===== 主循环: 遍历 K/V blocks =====
    for start_n in range(0, end_n, BLOCK_N):
        offs_n = start_n + tl.arange(0, BLOCK_N)

        # 加载 K block: [BLOCK_D, BLOCK_N] (转置)
        k_ptrs = (K + batch_id * stride_kb + head_id * stride_kh +
                  offs_d[:, None] * stride_kd + offs_n[None, :] * stride_kn)
        k = tl.load(k_ptrs,
                     mask=(offs_n[None, :] < seq_len) & (offs_d[:, None] < BLOCK_D),
                     other=0.0)

        # ===== GEMM①: Q * K^T =====
        # [BLOCK_M, BLOCK_D] × [BLOCK_D, BLOCK_N] → [BLOCK_M, BLOCK_N]
        qk = tl.dot(q, k) * sm_scale

        # Causal mask
        if IS_CAUSAL:
            mask = offs_m[:, None] >= offs_n[None, :]
            qk = tl.where(mask & (offs_n[None, :] < seq_len), qk, -1e8)
        else:
            qk = tl.where(offs_n[None, :] < seq_len, qk, -1e8)

        # Online softmax update
        m_ij = tl.maximum(m_i, tl.max(qk, 1))
        p = tl.exp(qk - m_ij[:, None])          # [BLOCK_M, BLOCK_N]
        l_ij = tl.sum(p, 1)                      # [BLOCK_M]

        alpha = tl.exp(m_i - m_ij)
        l_i = l_i * alpha + l_ij
        acc = acc * alpha[:, None]                # rescale 之前的累加

        # 加载 V block: [BLOCK_N, BLOCK_D]
        v_ptrs = (V + batch_id * stride_vb + head_id * stride_vh +
                  offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd)
        v = tl.load(v_ptrs,
                     mask=(offs_n[:, None] < seq_len) & (offs_d[None, :] < BLOCK_D),
                     other=0.0)

        # ===== GEMM②: attn * V =====
        # [BLOCK_M, BLOCK_N] × [BLOCK_N, BLOCK_D] → [BLOCK_M, BLOCK_D]
        acc += tl.dot(p.to(v.dtype), v)

        m_i = m_ij

    # Normalize
    acc = acc / l_i[:, None]

    # Store output
    out_ptrs = (Out + batch_id * stride_ob + head_id * stride_oh +
                offs_m[:, None] * stride_om + offs_d[None, :] * stride_od)
    tl.store(out_ptrs, acc.to(tl.float16),
             mask=(offs_m[:, None] < seq_len) & (offs_d[None, :] < BLOCK_D))

print("Prefill Attention Kernel 定义完成")
print()
print("两个关键 tl.dot:")
print("  1. qk = tl.dot(q, k) × sm_scale  → Q*K^T")
print("     shape: [BLOCK_M, BLOCK_D] × [BLOCK_D, BLOCK_N]")
print("     → head_dim 做 K 维度, 一次 tl.dot 覆盖整个 head_dim")
print()
print("  2. acc += tl.dot(p, v)            → attn*V")
print("     shape: [BLOCK_M, BLOCK_N] × [BLOCK_N, BLOCK_D]")
print("     → BLOCK_N 是 K 维度 (分块 softmax 的块大小)")

In [ ]:
def flash_attention_prefill(Q, K, V, causal=True):
    """Wrapper for prefill attention kernel"""
    B, H, S, D = Q.shape
    Out = torch.empty_like(Q)
    sm_scale = 1.0 / math.sqrt(D)

    BLOCK_M = 64
    BLOCK_N = 64
    BLOCK_D = triton.next_power_of_2(D)

    grid = (B, H, triton.cdiv(S, BLOCK_M))

    flash_attention_prefill_kernel[grid](
        Q, K, V, Out,
        sm_scale,
        Q.stride(0), Q.stride(1), Q.stride(2), Q.stride(3),
        K.stride(0), K.stride(1), K.stride(2), K.stride(3),
        V.stride(0), V.stride(1), V.stride(2), V.stride(3),
        Out.stride(0), Out.stride(1), Out.stride(2), Out.stride(3),
        S,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_D=BLOCK_D,
        IS_CAUSAL=causal,
    )
    return Out

# 验证
Q = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, dtype=torch.float16)
K = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, dtype=torch.float16)
V = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device, dtype=torch.float16)

# PyTorch reference
with torch.no_grad():
    ref = torch.nn.functional.scaled_dot_product_attention(Q, K, V, is_causal=True)

our = flash_attention_prefill(Q, K, V, causal=True)
diff = (our.float() - ref.float()).abs().max()
print(f"Triton FlashAttention vs PyTorch SDPA max diff: {diff:.6f}")
assert diff < 0.05, f"差异过大: {diff}"
print("✓ Prefill 验证通过")

## 8.3 Decode: Flash-Decoding 两阶段

Decode 阶段每次只生成 **1 个 token** (Q 长度=1)，但要与所有 KV cache 做注意力。

```
Decode 的矩阵形状:
  Q:   [1, head_dim]           ← 只有 1 个 token
  K:   [context_len, head_dim] ← 可能很长 (几万)
  V:   [context_len, head_dim]

  Q*K^T: [1, head_dim] × [head_dim, context_len] → [1, context_len]
  → 这是一个向量-矩阵乘法 (GEMV), 不是 GEMM!
  → 无法利用 Tensor Core 的矩阵并行

Flash-Decoding 解决方案:
  将 KV cache 分成 NUM_KV_SPLITS 段, 每段独立计算 partial attention,
  最后 reduce 合并结果.

  Stage 1: 每段计算 partial output + partial LSE
  Stage 2: 合并所有段的结果 (online softmax merge)
```

In [ ]:
@triton.jit
def decode_attention_stage1(
    Q, K_Buffer, V_Buffer,
    Att_Out,                    # [batch, heads, NUM_KV_SPLITS, head_dim+1]
    sm_scale,
    B_Seqlen,                   # [batch] 每个序列的长度
    stride_qbs, stride_qh,
    stride_kbs, stride_kh,
    stride_vbs, stride_vh,
    stride_mid_ob, stride_mid_oh, stride_mid_os,
    BLOCK_D: tl.constexpr,
    BLOCK_DV: tl.constexpr,
    BLOCK_N: tl.constexpr,
    NUM_KV_SPLITS: tl.constexpr,
    Lk: tl.constexpr,
    Lv: tl.constexpr,
):
    """
    Flash-Decoding Stage 1: 每个 KV split 独立计算 partial attention.
    对应 vLLM: vllm/v1/attention/ops/triton_decode_attention.py _fwd_kernel_stage1

    Grid: (batch, num_heads, NUM_KV_SPLITS)
    """
    cur_batch = tl.program_id(0)
    cur_head = tl.program_id(1)
    split_id = tl.program_id(2)

    cur_batch_seq_len = tl.load(B_Seqlen + cur_batch)

    # 计算当前 split 的范围
    kv_per_split = tl.cdiv(cur_batch_seq_len, NUM_KV_SPLITS)
    split_start = kv_per_split * split_id
    split_end = tl.minimum(split_start + kv_per_split, cur_batch_seq_len)

    # 加载 Q: [head_dim] (只有 1 个 token)
    offs_d = tl.arange(0, BLOCK_D)
    offs_dv = tl.arange(0, BLOCK_DV)
    mask_d = offs_d < Lk
    mask_dv = offs_dv < Lv

    q = tl.load(Q + cur_batch * stride_qbs + cur_head * stride_qh + offs_d,
                mask=mask_d, other=0.0)

    # Online softmax 状态
    e_max = -float("inf")
    e_sum = 0.0
    acc = tl.zeros([BLOCK_DV], dtype=tl.float32)

    if split_end > split_start:
        for start_n in range(split_start, split_end, BLOCK_N):
            offs_n = start_n + tl.arange(0, BLOCK_N)

            # 加载 K block: [BLOCK_N, head_dim]
            k = tl.load(K_Buffer + offs_n[:, None] * stride_kbs +
                        cur_head * stride_kh + offs_d[None, :],
                        mask=(offs_n[:, None] < split_end) & mask_d[None, :],
                        other=0.0)

            # ===== Q*K: 向量-矩阵乘 =====
            # [1, head_dim] × [head_dim, BLOCK_N] 但用 element-wise + reduce
            qk = tl.sum(q[None, :] * k, 1)  # [BLOCK_N]
            # ↑ 注意: 不是 tl.dot! 因为 Q 只有 1 行
            # vLLM 标准 decode kernel 也是用 tl.sum(q * k)
            qk *= sm_scale
            qk = tl.where(offs_n < split_end, qk, float("-inf"))

            # 加载 V block
            v = tl.load(V_Buffer + offs_n[:, None] * stride_vbs +
                        cur_head * stride_vh + offs_dv[None, :],
                        mask=(offs_n[:, None] < split_end) & mask_dv[None, :],
                        other=0.0)

            # Online softmax
            n_e_max = tl.maximum(tl.max(qk, 0), e_max)
            re_scale = tl.exp(e_max - n_e_max)
            p = tl.exp(qk - n_e_max)
            acc *= re_scale
            # ===== attn*V: 加权求和 =====
            acc += tl.sum(p[:, None] * v, 0)  # [BLOCK_DV]
            # ↑ 也不是 tl.dot! 因为输出只有 1 行
            e_sum = e_sum * re_scale + tl.sum(p, 0)
            e_max = n_e_max

        # 存储 partial output 和 LSE
        offs_out = (cur_batch * stride_mid_ob + cur_head * stride_mid_oh +
                    split_id * stride_mid_os)
        tl.store(Att_Out + offs_out + offs_dv, acc / e_sum, mask=mask_dv)
        tl.store(Att_Out + offs_out + Lv, e_max + tl.log(e_sum))

print("Decode Stage 1 定义完成")
print()
print("关键区别 — Decode 不用 tl.dot:")
print("  Q*K  = tl.sum(q[None, :] * k, 1)   # 向量内积, 非矩阵乘法")
print("  attn*V = tl.sum(p[:, None] * v, 0)  # 加权求和, 非矩阵乘法")
print()
print("原因: Q 只有 1 行, tl.dot 的 M 维度 = 1 无法利用 Tensor Core")
print("→ element-wise multiply + reduce 反而更高效")

## 8.4 GQA Grouped Kernel — 把 Decode GEMV 升级为 GEMM

vLLM 的核心优化: 当 `kv_group_num > 1` 时，将多个共享 KV head 的 Q head **批处理**。

In [ ]:
@triton.jit
def gqa_grouped_decode_kernel(
    Q, K_Buffer, V_Buffer,
    Att_Out,
    sm_scale,
    B_Seqlen,
    stride_qbs, stride_qh,
    stride_kbs, stride_kh,
    stride_vbs, stride_vh,
    stride_mid_ob, stride_mid_oh, stride_mid_os,
    kv_group_num: tl.constexpr,
    BLOCK_D: tl.constexpr,
    BLOCK_DV: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_H: tl.constexpr,        # 同时处理的 Q head 数
    NUM_KV_SPLITS: tl.constexpr,
    Lk: tl.constexpr,
    Lv: tl.constexpr,
):
    """
    GQA Grouped Decode Kernel — 多 Q head 一起做 GEMM.
    对应 vLLM: _fwd_grouped_kernel_stage1

    Grid: (batch, cdiv(num_q_heads, BLOCK_H), NUM_KV_SPLITS)
    """
    cur_batch = tl.program_id(0)
    cur_head_group = tl.program_id(1)
    split_id = tl.program_id(2)

    # 当前处理的 Q head 范围
    cur_head = cur_head_group * BLOCK_H + tl.arange(0, BLOCK_H)
    mask_h = cur_head < (cur_head_group + 1) * BLOCK_H

    # 对应的 KV head
    cur_kv_head = cur_head_group * BLOCK_H // kv_group_num

    cur_batch_seq_len = tl.load(B_Seqlen + cur_batch)
    kv_per_split = tl.cdiv(cur_batch_seq_len, NUM_KV_SPLITS)
    split_start = kv_per_split * split_id
    split_end = tl.minimum(split_start + kv_per_split, cur_batch_seq_len)

    offs_d = tl.arange(0, BLOCK_D)
    offs_dv = tl.arange(0, BLOCK_DV)
    mask_d = offs_d < Lk
    mask_dv = offs_dv < Lv

    # 加载 Q: [BLOCK_H, head_dim] — 多个 Q head!
    q_ptrs = (Q + cur_batch * stride_qbs +
              cur_head[:, None] * stride_qh + offs_d[None, :])
    q = tl.load(q_ptrs, mask=mask_h[:, None] & mask_d[None, :], other=0.0)

    e_max = tl.zeros([BLOCK_H], dtype=tl.float32) - float("inf")
    e_sum = tl.zeros([BLOCK_H], dtype=tl.float32)
    acc = tl.zeros([BLOCK_H, BLOCK_DV], dtype=tl.float32)

    if split_end > split_start:
        for start_n in range(split_start, split_end, BLOCK_N):
            offs_n = start_n + tl.arange(0, BLOCK_N)

            # K: [BLOCK_D, BLOCK_N] (转置后) — 只加载 1 个 KV head
            k = tl.load(K_Buffer + offs_d[:, None] +
                        offs_n[None, :] * stride_kbs +
                        cur_kv_head * stride_kh,
                        mask=mask_d[:, None] & (offs_n[None, :] < split_end),
                        other=0.0)

            # ===== GEMM: Q * K^T — 真正的矩阵乘法! =====
            # [BLOCK_H, BLOCK_D] × [BLOCK_D, BLOCK_N] → [BLOCK_H, BLOCK_N]
            qk = tl.dot(q, k.to(q.dtype))   # ← tl.dot! 利用 Tensor Core!
            qk *= sm_scale
            qk = tl.where(mask_h[:, None] & (offs_n[None, :] < split_end),
                          qk, float("-inf"))

            # V: [BLOCK_N, BLOCK_DV]
            v = tl.load(V_Buffer + offs_n[:, None] * stride_vbs +
                        cur_kv_head * stride_vh + offs_dv[None, :],
                        mask=(offs_n[:, None] < split_end) & mask_dv[None, :],
                        other=0.0)

            # Online softmax
            n_e_max = tl.maximum(tl.max(qk, 1), e_max)
            re_scale = tl.exp(e_max - n_e_max)
            p = tl.exp(qk - n_e_max[:, None])
            acc *= re_scale[:, None]

            # ===== GEMM: attn * V — 真正的矩阵乘法! =====
            # [BLOCK_H, BLOCK_N] × [BLOCK_N, BLOCK_DV] → [BLOCK_H, BLOCK_DV]
            acc += tl.dot(p.to(v.dtype), v)  # ← tl.dot!
            e_sum = e_sum * re_scale + tl.sum(p, 1)
            e_max = n_e_max

        # 存储 BLOCK_H 个 head 的结果
        offs_out = (cur_batch * stride_mid_ob +
                    cur_head[:, None] * stride_mid_oh +
                    split_id * stride_mid_os + offs_dv[None, :])
        tl.store(Att_Out + offs_out, acc / e_sum[:, None],
                 mask=mask_h[:, None] & mask_dv[None, :])

print("GQA Grouped Kernel 定义完成")
print()
print("核心升级: GEMV → GEMM")
print("  标准 decode: tl.sum(q * k)          → 向量运算")
print("  grouped:     tl.dot(q_block, k)     → 矩阵运算")
print()
print("  BLOCK_H 个 Q head 同时参与 tl.dot")
print("  → [BLOCK_H, head_dim] × [head_dim, BLOCK_N]")
print("  → 利用 Tensor Core 的 M 维度并行")
print()
print("典型配置: Llama-70B, kv_group_num=8, BLOCK_H=8")
print("  原来: 8 次向量点积")
print("  现在: 1 次 [8, 128] × [128, BLOCK_N] 矩阵乘法")

## 8.5 MLA 的 BLOCK_DPE: 双 tl.dot 分离机制

MLA (DeepSeek-V2/V3) 的 decode kernel 最大特殊性:
**Q*K 被拆成两次 tl.dot**，分别处理 nope 和 rope 维度。

```
标准 attention (GQA):
  Q: [BLOCK_H, head_dim]           head_dim = 128
  K: [head_dim, BLOCK_N]
  qk = tl.dot(Q, K)                 # 1 次 tl.dot

MLA attention:
  Q: [BLOCK_H, BLOCK_DMODEL + BLOCK_DPE]
     ├── q_nope: [BLOCK_H, 512]    # 吸收了 W_UK 的 nope 部分
     └── q_pe:   [BLOCK_H, 64]     # 经过 RoPE 的部分

  K in cache: [kv_lora_rank + rope_dim, BLOCK_N] = [576, BLOCK_N]
     ├── kv_c: [512, BLOCK_N]       # 压缩的 latent
     └── k_pe: [64, BLOCK_N]        # RoPE 部分

  # 为什么要分开？
  # 因为 kv_c 存储在 KV cache 中是连续的
  # 但 q_nope 和 q_pe 的计算路径不同:
  #   q_nope = (q_c @ W_UQ) @ W_UK  (吸收了解压权重)
  #   q_pe   = RoPE(q_c @ W_QR)      (经过位置编码)

  qk  = tl.dot(q_nope, kv_c)       # tl.dot ①: [BLOCK_H, 512] × [512, BLOCK_N]
  qk += tl.dot(q_pe, k_pe)         # tl.dot ②: [BLOCK_H, 64]  × [64, BLOCK_N]
  # 两部分相加得到完整的 attention score
```

In [ ]:
# MLA 的 BLOCK_DPE 在 vLLM 源码中的体现
# 源码: vllm/v1/attention/ops/triton_decode_attention.py:248-376

print("vLLM MLA Decode Kernel 中的双 tl.dot:")
print()
print("=== 加载 Q (两部分) ===")
print("""
# q_nope: [BLOCK_H, BLOCK_DMODEL]
offs_d = tl.arange(0, BLOCK_DMODEL)
q = tl.load(Q + cur_batch * stride_qbs +
            cur_head[:, None] * stride_qh + offs_d[None, :])

# q_pe: [BLOCK_H, BLOCK_DPE]
if BLOCK_DPE > 0:
    offs_dpe = BLOCK_DMODEL + tl.arange(0, BLOCK_DPE)
    qpe = tl.load(Q + cur_batch * stride_qbs +
                  cur_head[:, None] * stride_qh + offs_dpe[None, :])
""")

print("=== GEMM: 双 tl.dot ===")
print("""
for start_n in range(split_start, split_end, BLOCK_N):
    # K 的 nope 部分: [BLOCK_DMODEL, BLOCK_N]
    k = tl.load(K_Buffer + offs_d[:, None] + kv_loc[None, :] * stride_kbs)

    # tl.dot ①: Q_nope * K_nope
    qk = tl.dot(q, k.to(q.dtype))

    if BLOCK_DPE > 0:
        # K 的 rope 部分: [BLOCK_DPE, BLOCK_N]
        kpe = tl.load(K_Buffer + offs_dpe[:, None] + kv_loc[None, :] * stride_kbs)

        # tl.dot ②: Q_pe * K_pe
        qk += tl.dot(qpe, kpe.to(qpe.dtype))
""")

print("=== 参数配置 ===")
print("DeepSeek-V3 的典型值:")
print(f"  BLOCK_DMODEL = 512 (kv_lora_rank)")
print(f"  BLOCK_DPE = 64 (rope_head_dim)")
print(f"  总 head_dim = 512 + 64 = 576")
print(f"  Lk = 576 (KV cache 中存储的维度)")

## 8.6 三种 Decode Kernel 的 GEMM 对比

| 特征 | 标准 Decode | GQA Grouped | MLA Grouped |
|------|-----------|-------------|-------------|
| Q 形状 | `[1, d]` | `[BLOCK_H, d]` | `[BLOCK_H, 512+64]` |
| K 形状 | `[BLOCK_N, d]` | `[d, BLOCK_N]` | `[512, BLOCK_N] + [64, BLOCK_N]` |
| Q*K 方式 | `tl.sum(q * k)` | `tl.dot(q, k)` | `tl.dot(q, k) + tl.dot(qpe, kpe)` |
| 用 Tensor Core? | ✗ | ✓ | ✓ |
| tl.dot 次数/iter | 0 | 2 | 3 (含 attn*V) |
| head_dim | 128 | 128 | 576 |
| KV heads | 多个 | 1 个 (共享) | 1 个 (MQA) |

### 性能影响

```
标准 Decode (Q=[1, 128]):
  → 向量运算, 无法利用 Tensor Core
  → memory-bound (主要时间花在读 KV cache)

GQA Grouped (Q=[8, 128]):
  → 矩阵运算, Tensor Core 参与
  → 计算效率提升 ~2-4x (取决于 BLOCK_H)
  → 但仍然 memory-bound (KV cache 读取不变)

MLA Grouped (Q=[BLOCK_H, 576]):
  → head_dim 576 比标准 128 大 4.5x
  → 每次 tl.dot 的计算量更大
  → 但 KV cache 更小 (MQA, 只有 1 个 KV head)
  → 计算与内存的 trade-off
```

## 8.7 总结: 注意力中的 GEMM 全景

```
┌─────────────────────────────────────────────────────────────────┐
│                注意力 GEMM 全景                                 │
│                                                                 │
│  Prefill:                                                       │
│  ├── Q*K^T: tl.dot(q, k)    [BLOCK_M, d] × [d, BLOCK_N]      │
│  └── attn*V: tl.dot(p, v)   [BLOCK_M, BLOCK_N] × [BLOCK_N, d]│
│      → 真正的矩阵乘法, 利用 Tensor Core                       │
│      → FlashAttention 风格的 online softmax                    │
│                                                                 │
│  Decode (标准):                                                 │
│  ├── Q*K: tl.sum(q * k)     向量内积                           │
│  └── attn*V: tl.sum(p * v)  加权求和                           │
│      → 向量运算, 不用 Tensor Core                              │
│                                                                 │
│  Decode (GQA Grouped):                                          │
│  ├── Q*K: tl.dot(q, k)      [BLOCK_H, d] × [d, BLOCK_N]      │
│  └── attn*V: tl.dot(p, v)   [BLOCK_H, BLOCK_N] × [BLOCK_N, d]│
│      → 矩阵乘法! 多个 Q head 批处理                           │
│                                                                 │
│  Decode (MLA):                                                  │
│  ├── Q*K: tl.dot(q, k_nope) + tl.dot(qpe, kpe)   双 tl.dot   │
│  └── attn*V: tl.dot(p, v)                                      │
│      → rope/nope 分离, head_dim=576                            │
│                                                                 │
│  Paged KV Cache:                                                │
│  └── 所有 decode kernel 都需要页表查找 (Req_to_tokens)         │
│      → 间接寻址, 类似 MoE 的 sorted_token_ids                 │
└─────────────────────────────────────────────────────────────────┘
```

## 8.8 源码映射表

| 本章内容 | vLLM 源码位置 | 说明 |
|----------|--------------|------|
| Prefill Attention | `vllm/v1/attention/ops/triton_prefill_attention.py:37` | `_fwd_kernel` |
| Decode Stage 1 (标准) | `vllm/v1/attention/ops/triton_decode_attention.py:59` | `_fwd_kernel_stage1` |
| Decode Stage 1 (GQA) | `vllm/v1/attention/ops/triton_decode_attention.py:248` | `_fwd_grouped_kernel_stage1` |
| Decode Stage 2 | `vllm/v1/attention/ops/triton_decode_attention.py` | stage2 reduce kernel |
| Unified Attention | `vllm/v1/attention/ops/triton_unified_attention.py` | prefill+decode 统一 |
| MLA Triton Backend | `vllm/v1/attention/backends/mla/triton_mla.py` | TritonMLAImpl |
| MLA Decode Dispatch | `vllm/model_executor/layers/attention/mla_attention.py` | MLAAttention.forward |

### 下一章预告

> Notebook 09 将对比 vLLM 的 Triton GEMM、03_matmul_optimization 的终极 GEMM、
> 和 cuBLAS 的性能差距，分析根本原因。